# 03: Model Training
This notebook handles the environment setup, data ingestion, chronological splitting, and baseline model training for the FIFA World Cup 2026 prediction project.

In [3]:
import os
import pandas as pd
import numpy as np
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Define project paths
project_root = '/content/drive/My Drive/fifa-wc2026-predictor/'
training_dir = os.path.join(project_root, 'data_pipeline/03_model_training/')
input_dir = os.path.join(training_dir, 'input/')
output_dir = os.path.join(training_dir, 'output/')

# Create directory architecture
for path in [input_dir, output_dir]:
    os.makedirs(path, exist_ok=True)

print(f"Workspace established at: {training_dir}")

# Simple directory tree confirmation
def print_tree(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')

print_tree(training_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Workspace established at: /content/drive/My Drive/fifa-wc2026-predictor/data_pipeline/03_model_training/
/
input/
output/


## Consolidated Pipeline: Strict Data Ingestion, Calibrated Training, and Real Team Strengths

This cell implements the full pipeline as requested, ensuring data is loaded from the specified path, models are trained and calibrated, and team strengths are calculated using original country names. No dummy data will be generated or used.

In [4]:
import os
import pandas as pd
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss, roc_auc_score

# --- ✨  STEP 1: Strict Data Ingestion ---
print("\n--- STEP 1: Strict Data Ingestion ---")

# Explicitly use the exact, verified path for the master feature matrix
master_feature_matrix_path = '/content/drive/MyDrive/FIFA_Predictor_Data/final_engineered_feature_matrix.csv'

try:
    full_df = pd.read_csv(master_feature_matrix_path)
    print(f"Successfully loaded original feature matrix from: {master_feature_matrix_path}")
except FileNotFoundError:
    raise FileNotFoundError(f"Error: The specified file '{master_feature_matrix_path}' was not found. Halting execution as per strict constraints.")

# Convert 'date' column to datetime
full_df['date'] = pd.to_datetime(full_df['date'])

# --- Derive 'target' column from 'home_score' and 'away_score' ---
# 0: Away Win, 1: Draw, 2: Home Win
def get_match_outcome(row):
    if row['home_score'] > row['away_score']:
        return 2  # Home Win
    elif row['home_score'] < row['away_score']:
        return 0  # Away Win
    else:
        return 1  # Draw

full_df['target'] = full_df.apply(get_match_outcome, axis=1)
print("Derived 'target' column based on match outcomes.")

# Ensure 'home_team' and 'away_team' columns exist in the loaded dataframe
required_team_cols = ['home_team', 'away_team']
if not all(col in full_df.columns for col in required_team_cols):
    raise ValueError(f"Original feature matrix must contain {required_team_cols} columns for remapping.")

print(f"Data Shape: {full_df.shape}")
print("Data Head (showing real team names and features):")
display(full_df.head())

# --- ✨  STEP 2: Chronological Split & Calibrated Training ---
print("\n--- STEP 2: Chronological Split & Calibrated Training ---")

# Sort by date to ensure chronological split
full_df = full_df.sort_values('date').reset_index(drop=True)

total_rows = len(full_df)
train_idx = int(total_rows * 0.8)
val_idx = int(total_rows * 0.9)

# Preserve original dataframes for team name extraction later
train_df_full = full_df.iloc[:train_idx]
val_df_full = full_df.iloc[train_idx:val_idx]
test_df_full = full_df.iloc[val_idx:]

def report_split(name, data):
    print(f"{name} Split: {len(data)} records | {data['date'].min().date()} to {data['date'].max().date()}")

report_split("Train", train_df_full)
report_split("Validation", val_df_full)
report_split("Test", test_df_full)

# Define columns to drop for feature matrices (X_train, X_val)
# 'home_team' and 'away_team' are needed for remapping, not for model training features.
# 'home_score' and 'away_score' are used to derive 'target' and should not be features themselves.
drop_cols_for_features = ['date', 'target', 'home_team', 'away_team', 'home_score', 'away_score']

X_train = train_df_full.drop(columns=drop_cols_for_features, errors='ignore')
y_train = train_df_full['target']
X_val = val_df_full.drop(columns=drop_cols_for_features, errors='ignore')
y_val = val_df_full['target']

# Convert object columns to 'category' for XGBoost, ensuring categories are consistent
object_cols = X_train.select_dtypes(include='object').columns
for col in object_cols:
    # Get the categories from the training set
    train_categories = X_train[col].unique()
    # Create a CategoricalDtype based on training categories
    categorical_dtype = pd.CategoricalDtype(categories=train_categories)

    # Apply this CategoricalDtype to both X_train and X_val
    X_train[col] = X_train[col].astype(categorical_dtype)
    X_val[col] = X_val[col].astype(categorical_dtype)

print("Converted object columns to 'category' dtype for XGBoost.")

print("X_train features preview:")
display(X_train.head())
print("y_train target preview:")
display(y_train.head())

# Initialize XGBoost Classifier with aggressive regularization
xgb_model = XGBClassifier(
    objective='multi:softprob', # For multi-class probability output
    eval_metric='mlogloss',     # Multi-class logloss
    n_estimators=100,           # Number of boosting rounds
    max_depth=3,                # Strict depth constraint
    min_child_weight=5,         # Minimum sum of instance weight (hessian) needed in a child
    subsample=0.7,              # Subsample ratio of the training instance
    colsample_bytree=0.7,       # Subsample ratio of columns when constructing each tree
    reg_alpha=0.1,              # L1 regularization term on weights
    reg_lambda=0.1,             # L2 regularization term on weights
    use_label_encoder=False,    # Suppress warning for deprecated parameter
    enable_categorical=True,    # Enable categorical feature support in XGBoost
    random_state=42
)

# Wrap XGBoost model in CalibratedClassifierCV
print("Fitting and calibrating XGBoost model...")
calibrated_xgb = CalibratedClassifierCV(estimator=xgb_model, method='isotonic', cv=5)
calibrated_xgb.fit(X_train, y_train)
print("XGBoost model calibrated successfully.")

# Export the calibrated model
calibrated_xgb_model_path = os.path.join(output_dir, 'xgboost_calibrated_optimized.pkl')
joblib.dump(calibrated_xgb, calibrated_xgb_model_path)
print(f"Calibrated XGBoost model exported to: {calibrated_xgb_model_path}")

# Predict multi-class probabilities on the validation set
# Ensure val_probs is a numpy array for consistent indexing
val_probs_array = np.asarray(calibrated_xgb.predict_proba(X_val))
class_labels = calibrated_xgb.classes_ if hasattr(calibrated_xgb, 'classes_') else np.arange(val_probs_array.shape[1])

print("Validation set probabilities generated.")

# --- ✨  STEP 3: The True Simulation Bridge (Real Names) ---
print("\n--- STEP 3: The True Simulation Bridge (Real Names) ---")

# Create a mapping from class label (0, 1, 2) to its column index in val_probs_array
class_to_col_idx = {cls: idx for idx, cls in enumerate(class_labels)}

# Initialize probability arrays for each outcome to NaN
# These will be replaced with actual probabilities if the class exists
num_predictions = val_probs_array.shape[0]
p_away_win = np.full(num_predictions, np.nan)
p_draw = np.full(num_predictions, np.nan)
p_home_win = np.full(num_predictions, np.nan)

# Populate probabilities if the class is present in the model's output
if 0 in class_to_col_idx:
    p_away_win = val_probs_array[:, class_to_col_idx[0]]
if 1 in class_to_col_idx:
    p_draw = val_probs_array[:, class_to_col_idx[1]]
if 2 in class_to_col_idx:
    p_home_win = val_probs_array[:, class_to_col_idx[2]]

# Create DataFrame for predictions, using real team names from val_df_full
holdout_predictions_df_real_names = pd.DataFrame({
    'date': val_df_full['date'].reset_index(drop=True),
    'home_team': val_df_full['home_team'].reset_index(drop=True),
    'away_team': val_df_full['away_team'].reset_index(drop=True),
    'true_outcome': y_val.reset_index(drop=True),
    'P(Away_Win)': p_away_win,
    'P(Draw)': p_draw,
    'P(Home_Win)': p_home_win
})

print("Holdout Matrix with Real Names Preview:")
display(holdout_predictions_df_real_names.head())

# Aggregate Global Team Latent Strengths
team_strengths_real = {}

# Get all unique team names from the *entire* original dataset to ensure all 48 teams are covered
all_teams_in_original_df = pd.concat([full_df['home_team'], full_df['away_team']]).unique()

# Initialize strengths for all possible teams
for team in all_teams_in_original_df:
    team_strengths_real[team] = {'expected_win_probs': []}

# Iterate through the predictions (with real names) to aggregate strengths
for index, row in holdout_predictions_df_real_names.iterrows():
    home_team = row['home_team']
    away_team = row['away_team']
    p_home_win = row['P(Home_Win)']
    p_away_win = row['P(Away_Win)']

    # Ensure probabilities are not NaN before appending
    if not pd.isna(p_home_win):
        team_strengths_real[home_team]['expected_win_probs'].append(p_home_win)

    if not pd.isna(p_away_win):
        team_strengths_real[away_team]['expected_win_probs'].append(p_away_win)

final_team_strengths_real = []
for team, data in team_strengths_real.items():
    if data['expected_win_probs']:
        avg_win_prob = np.mean(data['expected_win_probs'])
        final_team_strengths_real.append({'team_name': team, 'avg_expected_win_probability': avg_win_prob})
    else:
        # If a team did not play in the validation set, its strength will be NaN
        # We can fill with a default (e.g., 0.0) or handle downstream if all 48 teams must have a value
        final_team_strengths_real.append({'team_name': team, 'avg_expected_win_probability': np.nan})

simulation_team_strengths_df_real_names = pd.DataFrame(final_team_strengths_real)

# Sort by average expected win probability for ranking
simulation_team_strengths_df_real_names = simulation_team_strengths_df_real_names.sort_values(
    by='avg_expected_win_probability', ascending=False
).reset_index(drop=True)

# Export the clean, fully mapped dataframe
simulation_output_path = os.path.join(output_dir, 'simulation_team_strengths.csv')
simulation_team_strengths_df_real_names.to_csv(simulation_output_path, index=False)
print(f"Updated simulation team strengths with real names saved to: {simulation_output_path}")

# --- ✨  STEP 4: Execution Output ---
print("\n--- STEP 4: True Top 10 International Contenders ---")

print("\nTrue Top 10 International Contenders (by avg_expected_win_probability):\n")
display(simulation_team_strengths_df_real_names.head(10).to_markdown(index=False))


--- STEP 1: Strict Data Ingestion ---
Successfully loaded original feature matrix from: /content/drive/MyDrive/FIFA_Predictor_Data/final_engineered_feature_matrix.csv
Derived 'target' column based on match outcomes.
Data Shape: (49619, 29)
Data Head (showing real team names and features):


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_rolling_form,...,home_squad_overall_max,away_squad_overall_max,home_squad_age_mean,away_squad_age_mean,form_difference,attack_vs_defense_clash,away_attack_vs_home_defense_clash,squad_quality_difference,neutral_venue_flag,target
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1.000000,...,76.0,86.0,24.675585,23.815706,0.000000,-1.0,-1.0,0.47233,0,1
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1.000000,...,86.0,76.0,23.815706,24.675585,0.000000,0.0,0.0,-0.47233,0,2
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,0.463316,...,76.0,86.0,24.675585,23.815706,-1.610052,0.0,0.0,0.47233,0,2
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1.076238,...,86.0,76.0,23.815706,24.675585,-0.607027,0.0,0.0,-0.47233,0,1
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1.373321,...,76.0,86.0,24.675585,23.815706,0.331667,0.0,0.0,0.47233,0,2



--- STEP 2: Chronological Split & Calibrated Training ---
Train Split: 39695 records | 1872-11-30 to 2015-11-17
Validation Split: 4962 records | 2015-11-17 to 2021-09-02
Test Split: 4962 records | 2021-09-02 to 2026-05-31
Converted object columns to 'category' dtype for XGBoost.
X_train features preview:


,tournament,city,country,neutral,home_rolling_form,home_rolling_goals_for,home_rolling_goals_against,home_rolling_clean_sheets,away_rolling_form,away_rolling_goals_for,...,away_squad_overall_mean,home_squad_overall_max,away_squad_overall_max,home_squad_age_mean,away_squad_age_mean,form_difference,attack_vs_defense_clash,away_attack_vs_home_defense_clash,squad_quality_difference,neutral_venue_flag
0,Friendly,Glasgow,Scotland,False,1.000000,0.500000,1.500000,0.100000,1.000000,0.500000,...,60.561115,76.0,86.0,24.675585,23.815706,0.000000,-1.0,-1.0,0.47233,0
1,Friendly,London,England,False,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,...,61.033445,86.0,76.0,23.815706,24.675585,0.000000,0.0,0.0,-0.47233,0
2,Friendly,Glasgow,Scotland,False,0.463316,1.073368,2.146736,0.463316,2.073368,2.146736,...,60.561115,76.0,86.0,24.675585,23.815706,-1.610052,0.0,0.0,0.47233,0
3,Friendly,London,England,False,1.076238,1.595244,1.519006,0.240497,1.683265,1.519006,...,61.033445,86.0,76.0,23.815706,24.675585,-0.607027,0.0,0.0,-0.47233,0
4,Friendly,Glasgow,Scotland,False,1.373321,1.737195,1.778850,0.131402,1.041655,1.778850,...,60.561115,76.0,86.0,24.675585,23.815706,0.331667,0.0,0.0,0.47233,0


y_train target preview:


...

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:40:22] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:40:33] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:40:44] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:40:49] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

XGBoost model calibrated successfully.
Calibrated XGBoost model exported to: /content/drive/My Drive/fifa-wc2026-predictor/data_pipeline/03_model_training/output/xgboost_calibrated_optimized.pkl
Validation set probabilities generated.

--- STEP 3: The True Simulation Bridge (Real Names) ---
Holdout Matrix with Real Names Preview:


,date,home_team,away_team,true_outcome,P(Away_Win),P(Draw),P(Home_Win)
0,2015-11-17,Rwanda,Libya,0,0.313986,0.272541,0.413473
1,2015-11-17,Senegal,Madagascar,2,0.072495,0.200346,0.727159
2,2015-11-17,Singapore,Syria,0,0.475499,0.235023,0.289478
3,2015-11-17,Slovakia,Iceland,2,0.376080,0.285451,0.338470
4,2015-11-17,Tunisia,Mauritania,2,0.087140,0.243559,0.669301


Updated simulation team strengths with real names saved to: /content/drive/My Drive/fifa-wc2026-predictor/data_pipeline/03_model_training/output/simulation_team_strengths.csv

--- STEP 4: True Top 10 International Contenders ---

True Top 10 International Contenders (by avg_expected_win_probability):



'| team_name        |   avg_expected_win_probability |\n|:-----------------|-------------------------------:|\n| Kernow           |                       0.762208 |\n| Gotland          |                       0.680097 |\n| Vatican City     |                       0.623629 |\n| Délvidék         |                       0.60835  |\n| Artsakh          |                       0.561657 |\n| Menorca          |                       0.557027 |\n| Belgium          |                       0.542542 |\n| Spain            |                       0.538619 |\n| Saint Barthélemy |                       0.538452 |\n| Brazil           |                       0.538163 |'

### 04: Publication-Grade Diagnostics Generation
This section programmatically generates high-fidelity visuals for the academic slide deck using the exported model artifacts and performance metrics.

In [7]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.calibration import CalibrationDisplay
from xgboost import XGBClassifier

# --- STEP 1: Programmatic Visuals Directory Setup ---
visuals_dir = os.path.join(training_dir, 'visuals/')
os.makedirs(visuals_dir, exist_ok=True)
print(f"Visuals directory mapped and ready: {visuals_dir}")

# --- STEP 2: Visual 7 — Master Model Comparison Plot ---
summary_path = os.path.join(output_dir, 'final_model_performance_summary.csv')

if os.path.exists(summary_path):
    perf_df = pd.read_csv(summary_path)
    fig, ax1 = plt.subplots(figsize=(12, 6), dpi=300)

    models = perf_df['Model'].values
    x = np.arange(len(models))
    width = 0.35

    color_baseline = '#1A237E'
    color_calibrated = '#C5A059'

    if 'ROC-AUC_Baseline' in perf_df.columns and 'ROC-AUC_Calibrated' in perf_df.columns:
        ax1.bar(x - width/2, perf_df['ROC-AUC_Baseline'], width, label='Baseline', color=color_baseline)
        ax1.bar(x + width/2, perf_df['ROC-AUC_Calibrated'], width, label='Calibrated', color=color_calibrated)

    ax1.set_xlabel('Model Architecture')
    ax1.set_ylabel('ROC-AUC Score')
    ax1.set_title('Model Performance Audit: Baseline vs. Calibrated', fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(models)
    ax1.set_ylim(0.45, 0.60)
    ax1.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(visuals_dir, '07_master_model_comparison.png'))
    plt.close()
    print("Visual 07 generated successfully.")

# --- STEP 3: Visual 8 — Calibration Reliability Diagram (Multi-class Adaptation) ---
model_path = os.path.join(output_dir, 'xgboost_calibrated_optimized.pkl')

if os.path.exists(model_path):
    calibrated_model = joblib.load(model_path)
    fig, ax = plt.subplots(figsize=(8, 8), dpi=300)

    # For multi-class, we visualize calibration for class 2 (Home Win) as a proxy
    y_val_binary = (y_val == 2).astype(int)
    val_probs = calibrated_model.predict_proba(X_val)[:, 2]

    CalibrationDisplay.from_predictions(
        y_val_binary,
        val_probs,
        n_bins=10,
        name='XGBoost (Home Win Class)',
        ax=ax,
        color='#2F4F4F'
    )

    ax.set_title("Rescuing Tree Overconfidence: Probability Calibration Curve", fontweight='bold')
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.savefig(os.path.join(visuals_dir, '08_probability_calibration_curve.png'))
    plt.close()
    print("Visual 08 generated successfully.")

# --- STEP 4: Visual 10 — Feature Importance Analysis ---
if hasattr(calibrated_model, 'calibrated_classifiers_'):
    base_xgb = calibrated_model.calibrated_classifiers_[0].estimator
    importances = base_xgb.feature_importances_
    feature_names = X_train.columns

    feat_imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
    feat_imp_df = feat_imp_df.sort_values(by='importance', ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(12, 8), dpi=300)
    ax.barh(feat_imp_df['feature'], feat_imp_df['importance'], color='#4682B4')
    ax.invert_yaxis()

    ax.set_xlabel('Importance Weight')
    ax.set_title('Pipeline Latent Strength: Predictive Signal Analysis', fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(visuals_dir, '10_feature_importance_analysis.png'))
    plt.close()
    print("Visual 10 generated successfully.")

Visuals directory mapped and ready: /content/drive/My Drive/fifa-wc2026-predictor/data_pipeline/03_model_training/visuals/
Visual 07 generated successfully.
Visual 08 generated successfully.
Visual 10 generated successfully.
